# Pneumonia Detection using a Neural Network from Scratch
## CMPE 49T: Fall 25 - Homework 4

**Overview**
In this assignment, we will implement a complete pipeline for binary classification of chest X-ray images to detect pneumonia using a custom-built neural network implemented in NumPy.

**Tasks**
1. Data Preparation and Exploration
2. Network Architecture
3. Training the Network
4. Evaluation and Reporting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import urllib.request
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# Set random seed for reproducibility
np.random.seed(42)

# Define dataset URL and filename
DATA_URL = "https://zenodo.org/record/10519652/files/pneumoniamnist.npz?download=1"
DATA_FILENAME = "pneumoniamnist.npz"

def download_data(url, filename):
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print("Download complete.")
    else:
        print(f"{filename} already exists.")

# Download the dataset
download_data(DATA_URL, DATA_FILENAME)

# Load the dataset
data = np.load(DATA_FILENAME)
print("Keys in the dataset:", data.files)

train_images = data['train_images']
train_labels = data['train_labels']
val_images = data['val_images']
val_labels = data['val_labels']
test_images = data['test_images']
test_labels = data['test_labels']

print(f"Train images shape: {train_images.shape}")
print(f"Train labels shape: {train_labels.shape}")
print(f"Val images shape: {val_images.shape}")
print(f"Val labels shape: {val_labels.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Test labels shape: {test_labels.shape}")

In [ ]:
# 1. Data Preparation and Exploration

# Normalize images to range [0, 1]
train_images_norm = train_images.astype('float32') / 255.0
val_images_norm = val_images.astype('float32') / 255.0
test_images_norm = test_images.astype('float32') / 255.0

# Visualize sample images
def visualize_samples(images, labels, class_names=['Normal', 'Pneumonia'], num_samples=5):
    plt.figure(figsize=(10, 4))
    for i in range(len(class_names)):
        # Find indices for the class
        indices = np.where(labels == i)[0]
        # Select random samples
        samples = np.random.choice(indices, num_samples, replace=False)
        
        for j, idx in enumerate(samples):
            plt.subplot(len(class_names), num_samples, i * num_samples + j + 1)
            plt.imshow(images[idx], cmap='gray')
            plt.title(class_names[i])
            plt.axis('off')
    plt.tight_layout()
    plt.show()

visualize_samples(train_images, train_labels)

In [ ]:
# Plot pixel intensity histograms
def plot_histograms(images, labels, class_names=['Normal', 'Pneumonia']):
    plt.figure(figsize=(12, 5))
    for i in range(len(class_names)):
        indices = np.where(labels == i)[0]
        class_images = images[indices]
        plt.subplot(1, 2, i + 1)
        plt.hist(class_images.ravel(), bins=50, color='blue' if i==0 else 'red', alpha=0.7)
        plt.title(f'Pixel Intensity Histogram - {class_names[i]}')
        plt.xlabel('Pixel Intensity')
        plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

plot_histograms(train_images_norm, train_labels)

# Compute and report class distributions
def report_distribution(labels, set_name):
    unique, counts = np.unique(labels, return_counts=True)
    dist = dict(zip(unique, counts))
    print(f"{set_name} Distribution: {dist}")
    return dist

train_dist = report_distribution(train_labels, "Train")
val_dist = report_distribution(val_labels, "Validation")
test_dist = report_distribution(test_labels, "Test")

# Calculate class weights
# Weight for class i = Total samples / (Number of classes * Samples in class i)
total_samples = len(train_labels)
n_classes = 2
class_weights = {}
for cls, count in train_dist.items():
    class_weights[cls] = total_samples / (n_classes * count)

print("Class Weights:", class_weights)

# Save normalized arrays
save_dir = "chatgpt_data_normalized"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

np.save(os.path.join(save_dir, "train_images.npy"), train_images_norm)
np.save(os.path.join(save_dir, "train_labels.npy"), train_labels)
np.save(os.path.join(save_dir, "val_images.npy"), val_images_norm)
np.save(os.path.join(save_dir, "val_labels.npy"), val_labels)
np.save(os.path.join(save_dir, "test_images.npy"), test_images_norm)
np.save(os.path.join(save_dir, "test_labels.npy"), test_labels)
print(f"Normalized data saved to {save_dir}/")

In [ ]:
# 2. Network Architecture

# Helper functions for Convolution and Pooling

def convolve2d(image, kernel):
    """
    Simple 2D convolution using valid padding.
    image: (H, W)
    kernel: (kH, kW)
    Returns: (H-kH+1, W-kW+1)
    """
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    
    output = np.zeros((out_H, out_W))
    
    for i in range(out_H):
        for j in range(out_W):
            output[i, j] = np.sum(image[i:i+kH, j:j+kW] * kernel)
            
    return output

def max_pooling2d(image, pool_size=2, stride=2):
    """
    Max pooling layer.
    image: (H, W)
    Returns: (H//pool_size, W//pool_size)
    """
    H, W = image.shape
    out_H = H // pool_size
    out_W = W // pool_size
    
    output = np.zeros((out_H, out_W))
    
    for i in range(out_H):
        for j in range(out_W):
            h_start = i * stride
            h_end = h_start + pool_size
            w_start = j * stride
            w_end = w_start + pool_size
            output[i, j] = np.max(image[h_start:h_end, w_start:w_end])
            
    return output

# Define Kernels
# Emboss Kernel
emboss_kernel = np.array([
    [-2, -1, 0],
    [-1, 1, 1],
    [0, 1, 2]
])

# Sobel Kernel (Horizontal and Vertical combined or just one? Usually Sobel is two kernels. 
# Instructions say "3x3 Sobel kernel". I'll use a standard one, maybe magnitude or just one direction.
# Let's use a combination or just one. I'll pick Sobel X for edge detection.)
# Actually, often "Sobel" implies edge detection. I'll use a standard edge detection filter or magnitude.
# Given "Convolution Layer 2: 3x3 Sobel kernel", singular. I'll use a generic edge detection or Sobel X.
sobel_kernel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])
sobel_kernel_y = np.array([
    [-1, -2, -1],
    [0, 0, 0],
    [1, 2, 1]
])
# Let's use Sobel X as the single kernel for simplicity as requested.
sobel_kernel = sobel_kernel_x

print("Kernels defined.")

# Activation Functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500))) # Clip to avoid overflow

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

# Loss Function: Weighted Binary Cross Entropy
def weighted_bce_loss(y_true, y_pred, weight_0, weight_1):
    # y_pred should be clipped to avoid log(0)
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    
    loss = - (weight_1 * y_true * np.log(y_pred) + weight_0 * (1 - y_true) * np.log(1 - y_pred))
    return np.mean(loss)

print("Activation and Loss functions defined.")

In [ ]:
# 3. Training the Network

# Initialize model
model = SimpleCNN()

# Pre-compute features for efficiency
print("Extracting features for training set...")
# Use tqdm for progress bar
X_train_features = []
for i in tqdm(range(train_images_norm.shape[0])):
    img = train_images_norm[i]
    # Conv1
    c1 = convolve2d(img, emboss_kernel)
    # Pool1
    p1 = max_pooling2d(c1)
    # Conv2
    c2 = convolve2d(p1, sobel_kernel)
    # Pool2
    p2 = max_pooling2d(c2)
    # Flatten
    f = p2.flatten()
    X_train_features.append(f)
X_train_features = np.array(X_train_features)

print("Extracting features for validation set...")
X_val_features = []
for i in tqdm(range(val_images_norm.shape[0])):
    img = val_images_norm[i]
    c1 = convolve2d(img, emboss_kernel)
    p1 = max_pooling2d(c1)
    c2 = convolve2d(p1, sobel_kernel)
    p2 = max_pooling2d(c2)
    f = p2.flatten()
    X_val_features.append(f)
X_val_features = np.array(X_val_features)

print("Extracting features for test set...")
X_test_features = []
for i in tqdm(range(test_images_norm.shape[0])):
    img = test_images_norm[i]
    c1 = convolve2d(img, emboss_kernel)
    p1 = max_pooling2d(c1)
    c2 = convolve2d(p1, sobel_kernel)
    p2 = max_pooling2d(c2)
    f = p2.flatten()
    X_test_features.append(f)
X_test_features = np.array(X_test_features)

print(f"Features shape: {X_train_features.shape}")

# Training Hyperparameters
epochs = 50
learning_rate = 0.01
batch_size = 32
patience = 5

# Class weights
w0 = class_weights[0]
w1 = class_weights[1]

train_losses = []
val_losses = []
val_accuracies = []
val_precisions = []
val_recalls = []
val_f1s = []

best_val_loss = float('inf')
patience_counter = 0

# Training Loop
for epoch in range(epochs):
    # Shuffle training data
    indices = np.arange(X_train_features.shape[0])
    np.random.shuffle(indices)
    X_train_shuffled = X_train_features[indices]
    y_train_shuffled = train_labels[indices]
    
    epoch_loss = 0
    num_batches = 0
    
    # Mini-batch SGD
    for i in range(0, X_train_features.shape[0], batch_size):
        X_batch = X_train_shuffled[i:i+batch_size]
        y_batch = y_train_shuffled[i:i+batch_size]
        
        loss = model.train_step(X_batch, y_batch, learning_rate, w0, w1)
        epoch_loss += loss
        num_batches += 1
        
    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)
    
    # Validation
    val_preds = model.predict(X_val_features)
    val_loss = weighted_bce_loss(val_labels, val_preds, w0, w1)
    val_losses.append(val_loss)
    
    # Metrics
    val_preds_binary = (val_preds > 0.5).astype(int)
    val_acc = accuracy_score(val_labels, val_preds_binary)
    val_prec = precision_score(val_labels, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels, val_preds_binary, zero_division=0)
    
    val_accuracies.append(val_acc)
    val_precisions.append(val_prec)
    val_recalls.append(val_rec)
    val_f1s.append(val_f1)
    
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
    
    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model weights (optional, here we just keep current)
        best_W1 = model.W1.copy()
        best_b1 = model.b1.copy()
        best_W2 = model.W2.copy()
        best_b2 = model.b2.copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# Restore best weights
model.W1 = best_W1
model.b1 = best_b1
model.W2 = best_W2
model.b2 = best_b2
print("Training complete.")

In [ ]:
# 3. Training the Network

# Initialize model
model = SimpleCNN()

# Pre-compute features for efficiency
print("Extracting features for training set...")
X_train_features = model.extract_features(train_images_norm)
print("Extracting features for validation set...")
X_val_features = model.extract_features(val_images_norm)
print("Extracting features for test set...")
X_test_features = model.extract_features(test_images_norm)

print(f"Features shape: {X_train_features.shape}")

# Training Hyperparameters
epochs = 50
learning_rate = 0.01
batch_size = 32
patience = 5

# Class weights
w0 = class_weights[0]
w1 = class_weights[1]

train_losses = []
val_losses = []
val_accuracies = []
val_precisions = []
val_recalls = []
val_f1s = []

best_val_loss = float('inf')
patience_counter = 0

# Training Loop
for epoch in range(epochs):
    # Shuffle training data
    indices = np.arange(X_train_features.shape[0])
    np.random.shuffle(indices)
    X_train_shuffled = X_train_features[indices]
    y_train_shuffled = train_labels[indices]
    
    epoch_loss = 0
    num_batches = 0
    
    # Mini-batch SGD
    for i in range(0, X_train_features.shape[0], batch_size):
        X_batch = X_train_shuffled[i:i+batch_size]
        y_batch = y_train_shuffled[i:i+batch_size]
        
        loss = model.train_step(X_batch, y_batch, learning_rate, w0, w1)
        epoch_loss += loss
        num_batches += 1
        
    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)
    
    # Validation
    val_preds = model.predict(X_val_features)
    val_loss = weighted_bce_loss(val_labels, val_preds, w0, w1)
    val_losses.append(val_loss)
    
    # Metrics
    val_preds_binary = (val_preds > 0.5).astype(int)
    val_acc = accuracy_score(val_labels, val_preds_binary)
    val_prec = precision_score(val_labels, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels, val_preds_binary, zero_division=0)
    
    val_accuracies.append(val_acc)
    val_precisions.append(val_prec)
    val_recalls.append(val_rec)
    val_f1s.append(val_f1)
    
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
    
    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model weights (optional, here we just keep current)
        best_W1 = model.W1.copy()
        best_b1 = model.b1.copy()
        best_W2 = model.W2.copy()
        best_b2 = model.b2.copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# Restore best weights
model.W1 = best_W1
model.b1 = best_b1
model.W2 = best_W2
model.b2 = best_b2
print("Training complete.")

In [ ]:
# 4. Evaluation and Reporting

# Plot Training and Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Plot Validation Metrics
plt.figure(figsize=(10, 5))
plt.plot(val_accuracies, label='Accuracy')
plt.plot(val_precisions, label='Precision')
plt.plot(val_recalls, label='Recall')
plt.plot(val_f1s, label='F1 Score')
plt.title('Validation Metrics over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Score')
plt.legend()
plt.show()

# Evaluate on Test Set
test_preds = model.predict(X_test_features)
test_preds_binary = (test_preds > 0.5).astype(int)

test_acc = accuracy_score(test_labels, test_preds_binary)
test_prec = precision_score(test_labels, test_preds_binary)
test_rec = recall_score(test_labels, test_preds_binary)
test_f1 = f1_score(test_labels, test_preds_binary)
test_auc = roc_auc_score(test_labels, test_preds)

# Specificity: TN / (TN + FP)
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds_binary).ravel()
specificity = tn / (tn + fp)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall: {test_rec:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Specificity: {specificity:.4f}")

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds_binary)
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Normal', 'Pneumonia'])
plt.yticks(tick_marks, ['Normal', 'Pneumonia'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, format(cm[i, j], 'd'),
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(test_labels, test_preds)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {test_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

# Discussion
print("Discussion:")
if test_rec < specificity:
    print("The model seems to struggle more with detecting Pneumonia (lower Recall).")
else:
    print("The model seems to struggle more with detecting Normal cases (lower Specificity).")
print(f"Class imbalance (Normal: {train_dist[0]}, Pneumonia: {train_dist[1]}) might have influenced this.")